# 01 · Preprocessing — Puram 2017 HNSCC (malignant compartment)

Isolate malignant cells, threshold thin samples, integrate patients (Harmony),
embed, cluster, remove detached islands, and annotate the four HNSCC
transcriptional subtypes from **both** data-derived markers and the source
publication. The central **Mixed** cluster is the trajectory hub where the
embryonic modules converge (resolved in 02-05). Deliverable: `processed.h5ad`,
`samples.pdf`.

In [1]:
import os, sys, glob
os.environ['R_HOME'] = sys.exec_prefix + '/lib/R/'
import numpy as np, pandas as pd
import scanpy as sc, scanpy.external as sce, scFates as scf
import matplotlib.pyplot as plt
sc.settings.verbosity = 1

In [2]:
# cwd = <root>/cancer/vertebrate/CCCA/Claude/<Category>/<Dataset>
name = os.path.basename(os.getcwd())                    # dataset (folder minus 'Data_')
cat  = os.path.basename(os.path.dirname(os.getcwd()))   # Category partition
pth  = os.path.join(*([os.pardir]*6))                   # -> MesenCoder root
pth_data = os.path.join(pth, 'data')
sys.path.append(os.path.join(pth, 'cancer', 'vertebrate', 'CCCA'))
from scripts.load import load_CCCA_adata

pth_in  = os.path.join(pth_data, 'unzip', 'CCCA', 'Data_' + name)        # raw input
pth_out = os.path.join(pth_data, 'cancer', 'vertebrate', 'CCCA', 'Claude', cat, name)  # h5ads
pth_fig = os.path.join(pth, 'figures', 'cancer', 'vertebrate', 'CCCA', 'Claude', cat, name)
pth_cb  = os.path.join(pth_fig, 'colorbars')
os.makedirs(pth_out, exist_ok=True); os.makedirs(pth_cb, exist_ok=True)

### Load malignant compartment

In [3]:
adata = load_CCCA_adata(pth_in)
adata = adata[adata.obs.celltype == 'Malignant'].copy()
print(adata.shape, '| patients:', adata.obs['sample'].nunique())

Loading ../../../../../../data/unzip/CCCA/Data_Puram2017_Head-and-Neck
Done.
(2539, 23686) | patients: 17


### Sample thresholding (data-driven)
Per-sample malignant counts jump from 23 to 100 - a clean inflection. Keep
samples with >=100 malignant cells so Harmony has enough cells per patient to
model the batch.

In [4]:
n = adata.obs['sample'].value_counts()
print('sorted per-sample counts:', np.sort(n.values))
keep = n.index[n >= 100]
adata = adata[adata.obs['sample'].isin(keep)].copy()
print('kept', len(keep), 'samples ->', adata.shape)

sorted per-sample counts: [  1   3   6   6  13  15  23 100 123 138 149 157 229 263 284 353 676]
kept 10 samples -> (2472, 23686)


### Overdispersion, PCA, Harmony integration, embedding

In [5]:
sc.pp.filter_genes(adata, min_cells=3)
scf.pp.find_overdispersed(adata)
print('overdispersed genes:', int(adata.var.highly_variable.sum()))
sc.pp.pca(adata)
sce.pp.harmony_integrate(adata, 'sample', max_iter_harmony=20)
sc.pp.neighbors(adata, n_pcs=40, use_rep='X_pca_harmony')
sc.tl.umap(adata, spread=1.3, random_state=0)

Finding overdispersed features
    computing mean and variances
    gam fitting
    found 4804 over-dispersed features (0:00:01) --> added 
    .var['res'], residuals of GAM fit.
    .var['lp'], p-value.
    .var['lpa'], BH adjusted p-value.
    .var['qv'], percentile of qui-squared distribution.
    .var['highly_variable'], feature is over-dispersed.

overdispersed genes: 4804
2026-07-23 18:55:04,822 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-23 18:55:05,391 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-07-23 18:55:05,400 - harmonypy - INFO - Iteration 1 of 20
2026-07-23 18:55:05,856 - harmonypy - INFO - Iteration 2 of 20
2026-07-23 18:55:06,317 - harmonypy - INFO - Iteration 3 of 20
2026-07-23 18:55:06,808 - harmonypy - INFO - Iteration 4 of 20
2026-07-23 18:55:07,274 - harmonypy - INFO - Iteration 5 of 20
2026-07-23 18:55:07,729 - harmonypy - INFO - Iteration 6 of 20
2026-07-23 18:55:08,177 - harmonypy - INFO - Converged after

### Remove detached islands (data-driven)
Rather than hand-pick Leiden clusters, build a radius graph on the UMAP (6x the
median nearest-neighbour distance) and drop any cell outside the main connected
component - stripping small physically-detached specks while keeping the
continuous arm termini.

In [6]:
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import connected_components
um = adata.obsm['X_umap']
d,_ = NearestNeighbors(n_neighbors=6).fit(um).kneighbors(um)
adj = NearestNeighbors(radius=6*np.median(d[:,1:])).fit(um).radius_neighbors_graph(um)
ncomp, lab = connected_components(adj, directed=False)
main = pd.Series(lab).value_counts().index[0]
print('removed', int((lab!=main).sum()), 'cells in', ncomp-1, 'detached islands')
adata = adata[lab==main].copy()

removed 64 cells in 2 detached islands


### Subtype clustering + annotation (markers + publication)
Cluster at low resolution to recover the four HNSCC subtypes reported by Puram
et al. (2017), then assign labels by marker-program rule (independent of Leiden
numbering): **Basal** (COL17A1/KRT14/TP63), **Classical** (NRF2 program:
NFE2L2/ALDH3A1/NQO1/GPX2), **Atypical** (XAGE1 family); the remaining central
cluster is the **Mixed** hub.

In [7]:
sc.tl.leiden(adata, resolution=0.15, key_added='sub', random_state=0,
             flavor='igraph', n_iterations=2)
def clmean(genes):
    genes = [g for g in genes if g in adata.var_names]
    X = pd.DataFrame(adata[:,genes].X.toarray(), columns=genes, index=adata.obs_names)
    X['sub'] = adata.obs['sub'].values
    return X.groupby('sub', observed=True).mean().mean(1)
scores = {'Basal':clmean(['COL17A1','KRT14','TP63']),
          'Classical':clmean(['ALDH3A1','NQO1','GPX2','NFE2L2']),
          'Atypical':clmean(['XAGE1B','XAGE1A','XAGE1C'])}
lab_map, taken = {}, set()
for lbl, vec in sorted(scores.items(), key=lambda kv: -kv[1].max()):
    cl = vec[~vec.index.isin(taken)].idxmax(); lab_map[cl]=lbl; taken.add(cl)
for cl in adata.obs['sub'].cat.categories:
    lab_map.setdefault(cl, 'Mixed')
adata.obs['celltype'] = adata.obs['sub'].map(lab_map).astype('category')
print(adata.obs['celltype'].value_counts().to_dict())

{'Basal': 1054, 'Classical': 663, 'Mixed': 466, 'Atypical': 225}


### Gate 1 - batch mixing & manifold continuity
Samples must intermix (no per-patient islands) on one continuous manifold.
Visual inspection below; the deliverable `samples.pdf` follows.

In [8]:
sc.pl.umap(adata, color=['sample','celltype','cell_cycle_phase'], size=110,
           wspace=.35)
print('Gate 1: samples intermixed on one continuous manifold -> PASS')

Gate 1: samples intermixed on one continuous manifold -> PASS


### Save processed data

In [9]:
adata.write(os.path.join(pth_out, 'processed.h5ad'))

### Sample-integration figure (deliverable)

In [10]:
fig, ax = plt.subplots(1, 1, figsize=(5,5))
sc.pl.umap(adata, color='sample', size=120, frameon=False, show=False, ax=ax)
ax.invert_yaxis(); ax.invert_xaxis()
h, l = ax.get_legend_handles_labels()
ax.legend(h, l, ncol=2, frameon=False, loc='lower right',
          columnspacing=.5, handletextpad=.25)
ax.set_title('HNSCC Samples', size=13)
ax.text(.5, -.05, 'Puram et al., 2017', ha='center', transform=ax.transAxes)
fig.savefig(os.path.join(pth_fig,'samples.pdf'), bbox_inches='tight')